# Python で AFM 像を解析する

このノートブックでは、AFM Nanofiber Analyzer の解析を GUI を使わずに Python から
実行します。AFM の高さ像を解析して `.b2z` バンドルを作り、補正後の高さ像に
スケルトンとキンクを重ねて描き、すべての繊維を計測して表にし、その表を CSV に
保存し、最後に複数の像をまとめて解析・計測します。

GUI と `cli.py` も同じ関数（`afm_nanofiber_analyzer.pipeline` と
`afm_nanofiber_analyzer.measure`）を呼ぶので、入力と設定が同じなら、ここでも同じ
数値になります。

## 準備

このパッケージは PyPI では公開していません。リポジトリをチェックアウトした
フォルダからインストールし、続けて Jupyter を入れます。

```
python -m pip install .
python -m pip install jupyterlab
```

`pip install .` で入るのは解析パッケージ `afm_nanofiber_analyzer` だけです。GUI と
`cli.py` はチェックアウトしたフォルダから実行します。

## 入力

下のパスは、このノートブックのフォルダ（チェックアウトの
`docs/manual/notebooks/`）からの相対パスで、`testdata/` に同梱したスキャンを
指しています。自分のファイルに置き換えてください。

走査範囲は長さの単位を決めます。島津のテキスト出力では、`None` にするとヘッダ
（`SizeX` / `SizeY`）から読みます。ヘッダに記録されていないファイルでは、ここで
走査範囲を与えてください。与えないとバンドルに走査範囲が記録されず、
`measure_bundle` がエラーで止まります。

In [ ]:
from pathlib import Path

# Input scans with their scan size (x_um, y_um). None reads the size from the
# file header; a text export whose header has none needs it given here.
DATA_DIR = Path("../../../testdata")
INPUTS = [
    (DATA_DIR / "TunicateACTOCCNF.txt", (5.0, 5.0)),
    (DATA_DIR / "_20250318-164122_T.ssp .txt", None),
]

# Folder the bundles and CSV files are written to.
OUTPUT_DIR = Path("afm_output")
OUTPUT_DIR.mkdir(exist_ok=True)

## インポート

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from afm_nanofiber_analyzer.blosc2_io import load_bundle, load_bundle_meta
from afm_nanofiber_analyzer.bundle_schema import pixel_size_nm, scan_size_um_from_meta
from afm_nanofiber_analyzer.measure import collect_fiber_stats, measure_bundle, write_fiber_csv
from afm_nanofiber_analyzer.pipeline import ProcParams, process_file

## 解析の設定

`ProcParams` は解析の設定をすべて持ち、既定値は GUI01 と同じです。設定を変える
ときは、たとえば `ProcParams(bg_method="tophat")` のように引数で渡します。設定は
バンドルの横に `<入力ファイル名>_param.json` として書き出されます。各設定の働きは
[`docs/algorithms.ja.md`](../../algorithms.ja.md) で説明しています。

In [ ]:
params = ProcParams()
dataclasses.asdict(params)

## 1 枚の像を解析する

`process_file` は背景補正・二値化・細線化・キンク検出を行い、
`<入力ファイル名>.b2z` と `<入力ファイル名>_param.json` を `OUTPUT_DIR` に書き出し
ます。2 つの警告リストは、背景補正や画素ノイズが繊維の高さに影響した可能性が
ある理由を示します。該当しなければ空です。

In [ ]:
input_path, scan_size_um = INPUTS[0]
result = process_file(
    str(input_path), params, output_dir=str(OUTPUT_DIR), scan_size_um=scan_size_um,
)
print("bundle:", result.bundle_path)
print("parameters:", result.param_path)
print(f"elapsed: {result.elapsed_s:.1f} s")
print("background warnings:", result.background_warnings)
print("noise warnings:", result.noise_warnings)

## 繊維を計測する

`measure_bundle` はバンドル内のすべての繊維を追跡し、繊維ごとに `FiberStats` を
1 つ返します。長さ、高さ、幅、キンクの数と角度などです。各値の計算方法は
[`docs/gui04_measurements.ja.md`](../../gui04_measurements.ja.md) で説明しています。

この呼び方では、解析したままのバンドルを計測します。除外する繊維はなく、断片の
連結もしません。

In [ ]:
measured = measure_bundle(result.bundle_path)
fibers = pd.DataFrame([dataclasses.asdict(s) for s in measured.stats])
print(f"{len(fibers)} fibers, scan size {measured.scale_um} x {measured.scale_y_um} µm "
      f"({measured.scale_source})")
fibers.head()

## 結果を見る

バンドルの配列は `load_bundle` で読みます。`calibrated` は背景補正後の高さ（nm）
です。サンプリング間隔はバンドルに記録された走査範囲から `pixel_size_nm` で求める
ので、軸の単位は µm です。

右の図では、計測した各繊維を、値を計測した中心線（`Fiber.xtrack`、
`Fiber.ytrack`）とキンク（`Fiber.kink_indices`）で描いています。像の上への
置き方は GUI04 と同じです。

In [ ]:
height = load_bundle(result.bundle_path, keys=["calibrated"])["calibrated"]
scan_x_um, scan_y_um = scan_size_um_from_meta(load_bundle_meta(result.bundle_path))
rows, cols = height.shape
px_x_um = pixel_size_nm(scan_x_um, cols) / 1000
px_y_um = pixel_size_nm(scan_y_um, rows) / 1000
extent = (0, cols * px_x_um, rows * px_y_um, 0)
vmin, vmax = np.percentile(height, [1, 99])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
im = ax1.imshow(height, cmap="afmhot", vmin=vmin, vmax=vmax, extent=extent)
ax1.set_title("Calibrated height")
fig.colorbar(im, ax=ax1, label="Height (nm)")

ax2.imshow(height, cmap="gray", vmin=vmin, vmax=vmax, extent=extent)
for i, fiber in enumerate(measured.fibers):
    # The centerline is in pixels from the corner of the fiber's bounding box,
    # which fiber.data[0] (x) and fiber.data[1] (y) give.
    x = (fiber.xtrack + fiber.data[0] + 0.5) * px_x_um
    y = (fiber.ytrack + fiber.data[1] + 0.5) * px_y_um
    ax2.plot(x, y, color="deepskyblue", linewidth=1.2,
             label="Centerline" if i == 0 else None)
    kinks = np.asarray(fiber.kink_indices, dtype=int)
    ax2.scatter(x[kinks], y[kinks], s=60,
                facecolors="none", edgecolors="orange", linewidths=1.5,
                label="Kink" if i == 0 else None)
ax2.set_title("Measured centerlines and kinks")
ax2.legend(loc="upper right")
for ax in (ax1, ax2):
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
plt.show()

## 表を保存する

`write_fiber_csv` は `cli.py measure` と同じ列を書き出します。ファイルは BOM 付き
UTF-8 なので、日本語 Windows の Excel でも文字化けせずに開けます。

In [ ]:
csv_path = OUTPUT_DIR / (Path(result.bundle_path).stem + "_fibers.csv")
write_fiber_csv(
    str(csv_path), measured.stats, scale_um=measured.scale_um,
    scale_y_um=measured.scale_y_um, scale_source=measured.scale_source,
)
print("written:", csv_path)

## 分布

走査範囲の端に達している繊維（`touches_frame`）や、端が交差で切れている繊維
（`cut_end_count`）は、計測した範囲の外へ続いているので、`length_nm` は下限値に
すぎません。下の長さのヒストグラムでは、そうした繊維を分けて積み上げています。
このような繊維の補正は GUI03 にあります。
[`docs/gui03_statistics.ja.md`](../../gui03_statistics.ja.md) を参照してください。

In [ ]:
lower_bound = fibers["touches_frame"].astype(bool) | (fibers["cut_end_count"] > 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
ax1.hist([fibers.loc[~lower_bound, "length_nm"], fibers.loc[lower_bound, "length_nm"]],
         bins=20, stacked=True, label=["Both ends are fiber ends", "Length is a lower bound"])
ax1.set_xlabel("Length (nm)")
ax1.set_ylabel("Fibers")
ax1.legend()
ax2.hist(fibers["height_median_nm"].dropna(), bins=20)
ax2.set_xlabel("Median height (nm)")
ax2.set_ylabel("Fibers")
plt.show()

## 複数の像

`collect_fiber_stats` はバンドルのリストを計測し、バンドルごとの統計値を返します。
計測できなかったバンドルとその理由も一緒に返します。

`apply_exclusions=True` と `apply_connection=True` を付けると、GUI04 がバンドルの
横に保存したもの、つまり除外した繊維（`<stem>_excluded.json`）と連結した断片
（`<stem>_connect.json`）を適用します。これらのファイルが無いバンドルは、ここでの
ように解析したまま計測します。連結ファイルを保存した後にバンドルを解析し直した
ためにスケルトンが変わっている場合、その連結ファイルは拒否され、`errors` に
載ります。

In [ ]:
bundle_paths = [result.bundle_path]
for input_path, scan_size_um in INPUTS[1:]:
    bundle_paths.append(process_file(
        str(input_path), params, output_dir=str(OUTPUT_DIR), scan_size_um=scan_size_um,
    ).bundle_path)

per_bundle, errors = collect_fiber_stats(
    bundle_paths, apply_exclusions=True, apply_connection=True,
)
for path, message in errors:
    print("failed:", path, message)

all_fibers = pd.concat(
    [pd.DataFrame([dataclasses.asdict(s) for s in stats]).assign(bundle=Path(path).name)
     for path, stats in per_bundle],
    ignore_index=True,
)
all_fibers.groupby("bundle")[["length_nm", "height_median_nm"]].agg(["count", "median"])